# CogMem — BigCodeBench Evaluation

Run models on BigCodeBench (1140 tasks). Collect episodes with success/failure + model reasoning.

**Models:** Set `MODEL` variable in Cell 6 to switch between:
- `qwen2.5-coder:3b` — base coder model (baseline)
- `cogmem-qwen-bigcode` — CogMem DoRA model

**Flow:** Cells 1-6 sequentially. Resume-safe.

In [1]:
# Cell 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

NVIDIA RTX A4000, 16376 MiB, 16101 MiB


In [2]:
# Cell 2: Install system deps + Ollama (DO NOT touch torch)
!apt-get update -qq && apt-get install -y -qq zstd cmake build-essential > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
!pip install "transformers==4.43.4" "peft==0.12.0" "accelerate==0.33.0" "datasets>=2.20" "huggingface-hub>=0.24" pyyaml -q
!pip install openai -q

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to render group...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
# Cell 3: Start Ollama + pull model
import subprocess, time, os
proc = subprocess.Popen(
    ["ollama", "serve"],
    env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"},
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)
!ollama pull qwen2.5-coder:3b
print("Ollama + Qwen2.5-Coder:3b ready!")

In [ ]:
# Run this to check
from openai import OpenAI
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# What model is loaded?
!ollama list

# Test the model
resp = client.chat.completions.create(
    model="cqwen2.5:3b",
    messages=[{"role": "user", "content": "Write a Python function that adds two numbers."}],
    max_tokens=200, temperature=0,
)
print(resp.choices[0].message.content[:500])


]11;?\NAME                          ID              SIZE      MODIFIED      
qwen2.5:3b                    357c53fb659c    1.9 GB    6 seconds ago    
cogmem-qwen-bigcode:latest    4abe6c6f0baa    6.2 GB    2 hours ago      


In [ ]:
# Cell 4: Clone CogMem + load BigCodeBench full dataset (1140 tasks)
!cd /notebooks && git clone https://github.com/tungooxx/CogMem.git 2>/dev/null || (cd /notebooks/CogMem && git pull && git checkout feat/bigcodebench-integration)
!cd /notebooks/CogMem && pip install -e . --no-deps -q

import sys
if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

import cogmem
print(f"cogmem loaded from {cogmem.__file__}")

from datasets import load_dataset

# BigCodeBench full: 1140 tasks
ds = load_dataset("bigcode/bigcodebench", split="v0.1.4")
print(f"BigCodeBench: {len(ds)} tasks")

tasks = []
for item in ds:
    tasks.append({
        "task_id": item["task_id"],
        "instruct_prompt": item.get("instruct_prompt", ""),
        "complete_prompt": item.get("complete_prompt", ""),
        "test": item.get("test", ""),
        "canonical_solution": item.get("canonical_solution", ""),
        "entry_point": item.get("entry_point", ""),
    })

import json
with open("/notebooks/bigcodebench_tasks.jsonl", "w") as f:
    for t in tasks:
        f.write(json.dumps(t) + "\n")
print(f"Saved {len(tasks)} tasks")

In [ ]:
# Cell 5: Quick sanity check
import importlib
import cogmem.benchmarks.bigcodebench.evaluator
import cogmem.benchmarks.bigcodebench.prompts
importlib.reload(cogmem.benchmarks.bigcodebench.evaluator)
importlib.reload(cogmem.benchmarks.bigcodebench.prompts)

from openai import OpenAI
from cogmem.benchmarks.bigcodebench.prompts import format_messages, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

for task in tasks[:3]:
    messages = format_messages(task, use_instruct=True)
    resp = client.chat.completions.create(
        model="qwen2.5-coder:3b", messages=messages,
        max_tokens=2048, temperature=0,
    )
    response = resp.choices[0].message.content
    code = extract_code(response)
    result = evaluate_solution(task, code, timeout=30, mode="subprocess")
    status = "PASS" if result["passed"] else "FAIL"
    print(f"{task['task_id']}: {status}")
    if not result["passed"] and result.get("error"):
        print(f"  Error: {result['error'][:200]}")
    print()

print("Sanity check done!")

In [ ]:
# Cell 6: Run BigCodeBench full evaluation (resume-safe)
# ~2-3 hours for 1140 tasks on A4000
#
# CHANGE THIS to switch models:
MODEL = "qwen2.5-coder:3b"  # baseline
# MODEL = "cogmem-qwen-bigcode"  # after Phase 2 training

import json, time
from pathlib import Path
from openai import OpenAI
from cogmem.benchmarks.bigcodebench.prompts import format_messages, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution

CHECKPOINT = f"/notebooks/bigcode_full_{MODEL.replace(':', '_').replace('-', '_')}.jsonl"

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

completed_ids = set()
episodes = []
if Path(CHECKPOINT).exists():
    with open(CHECKPOINT, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                ep = json.loads(line)
                completed_ids.add(ep["task_id"])
                episodes.append(ep)
            except json.JSONDecodeError:
                break
    print(f"Resuming: {len(completed_ids)} tasks already done")

tasks = []
with open("/notebooks/bigcodebench_tasks.jsonl") as f:
    for line in f:
        tasks.append(json.loads(line.strip()))

remaining = [t for t in tasks if t["task_id"] not in completed_ids]
total = len(tasks)
done = len(completed_ids)
passed = sum(1 for ep in episodes if ep["success"])
start_time = time.time()

print(f"Model: {MODEL}")
print(f"Total: {total}, Done: {done}, Remaining: {len(remaining)}")
print(f"Current pass rate: {passed}/{done} = {passed/max(done,1):.1%}")
print("=" * 60)

for i, task in enumerate(remaining):
    try:
        messages = format_messages(task, use_instruct=True)
        resp = client.chat.completions.create(
            model=MODEL, messages=messages,
            max_tokens=2048, temperature=0,
        )
        response = resp.choices[0].message.content
        code = extract_code(response)
        result = evaluate_solution(task, code, timeout=30, mode="subprocess")
        episode = {
            "episode_id": f"bigcode_{task['task_id'].replace('/', '_')}_{int(time.time())}",
            "task_id": task["task_id"],
            "task_type": "bigcodebench",
            "task_description": task.get("instruct_prompt", task.get("complete_prompt", "")),
            "script": response,
            "generated_code": code,
            "success": result["passed"],
            "q_value": 1.0 if result["passed"] else -1.0,
            "error": result.get("error"),
            "entry_point": task.get("entry_point", ""),
            "model": MODEL,
            "timestamp": time.time(),
        }
    except Exception as e:
        episode = {
            "episode_id": f"bigcode_{task['task_id'].replace('/', '_')}_{int(time.time())}",
            "task_id": task["task_id"],
            "task_type": "bigcodebench",
            "task_description": task.get("instruct_prompt", ""),
            "script": "", "generated_code": "",
            "success": False, "q_value": -1.0,
            "error": str(e),
            "entry_point": task.get("entry_point", ""),
            "model": MODEL,
            "timestamp": time.time(),
        }

    episodes.append(episode)
    done += 1
    if episode["success"]: passed += 1

    with open(CHECKPOINT, "a", encoding="utf-8") as f:
        f.write(json.dumps(episode, ensure_ascii=False) + "\n")

    status = "PASS" if episode["success"] else "FAIL"
    elapsed = time.time() - start_time
    rate = (i + 1) / elapsed * 3600 if elapsed > 0 else 0
    eta = (len(remaining) - i - 1) / rate * 3600 if rate > 0 else 0

    if (i + 1) % 10 == 0 or i < 5:
        print(f"[{done}/{total}] {task['task_id']}: {status} | "
              f"Pass: {passed}/{done} ({passed/done:.1%}) | "
              f"Rate: {rate:.0f}/hr | ETA: {eta/60:.0f}m")

print("=" * 60)
print(f"DONE! {MODEL}: {passed}/{done} passed ({passed/done:.1%})")
print(f"Checkpoint: {CHECKPOINT}")

In [ ]:
# Cell 6b: Q-value distribution + episode analysis
import json
from pathlib import Path
from collections import Counter

CHECKPOINT = f"/notebooks/bigcode_full_{MODEL.replace(':', '_').replace('-', '_')}.jsonl"

episodes = []
with open(CHECKPOINT, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            episodes.append(json.loads(line))

total = len(episodes)
passed = sum(1 for ep in episodes if ep["success"])
failed = total - passed
q_vals = [ep["q_value"] for ep in episodes]

print(f"{'='*60}")
print(f"Model: {MODEL} — BigCodeBench Full")
print(f"{'='*60}")
print(f"Total: {total}, Passed: {passed} ({passed/total:.1%}), Failed: {failed}")
print(f"Q-values: min={min(q_vals):.1f}, max={max(q_vals):.1f}, mean={sum(q_vals)/len(q_vals):.2f}")

# Q-value distribution
high = sum(1 for q in q_vals if q >= 0.7)
mid = sum(1 for q in q_vals if 0.3 <= q < 0.7)
low = sum(1 for q in q_vals if q < 0.3)
print(f"\nQ-value zones:")
print(f"  High  (Q >= 0.7):  {high:>4} ({high/total:.1%})")
print(f"  Mid   (0.3-0.7):   {mid:>4} ({mid/total:.1%})")
print(f"  Low   (Q < 0.3):  {low:>4} ({low/total:.1%})")

# Error breakdown
print(f"\nError breakdown:")
errors = Counter()
for ep in episodes:
    if not ep["success"] and ep.get("error"):
        err = ep["error"][:150]
        if "Timeout" in err:
            errors["Timeout"] += 1
        elif "SyntaxError" in err:
            errors["SyntaxError"] += 1
        elif "ImportError" in err or "ModuleNotFoundError" in err:
            errors["ImportError"] += 1
        elif "NameError" in err:
            errors["NameError"] += 1
        elif "TypeError" in err:
            errors["TypeError"] += 1
        elif "AttributeError" in err:
            errors["AttributeError"] += 1
        elif "AssertionError" in err or "FAIL:" in err:
            errors["Wrong answer"] += 1
        else:
            errors["Other runtime"] += 1

for err_type, count in errors.most_common():
    print(f"  {err_type:<20} {count:>4} ({count/failed:.1%} of failures)")

# Per-task results
print(f"\nPassed tasks:")
for ep in episodes:
    if ep["success"]:
        print(f"  {ep['task_id']}: Q={ep['q_value']:.1f}")

print(f"\nSample failures (first 5):")
fail_count = 0
for ep in episodes:
    if not ep["success"] and fail_count < 5:
        err_short = (ep.get("error") or "no error")[:80]
        print(f"  {ep['task_id']}: {err_short}")
        fail_count += 1

In [ ]:
# Cell 7: Show results for all models evaluated
import json
from pathlib import Path

print("=" * 60)
print("BigCodeBench Full Results")
print("=" * 60)

for ckpt in sorted(Path("/notebooks").glob("bigcode_full_*.jsonl")):
    episodes = []
    with open(ckpt, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                episodes.append(json.loads(line))
    if not episodes: continue
    total = len(episodes)
    passed = sum(1 for ep in episodes if ep["success"])
    model = episodes[0].get("model", ckpt.stem)
    print(f"  {model}: {passed}/{total} ({passed/total:.1%})")

print("=" * 60)

In [ ]:
# Cell 8: Save episodes as memory bank
import json
from pathlib import Path

CHECKPOINT = f"/notebooks/bigcode_full_{MODEL.replace(':', '_').replace('-', '_')}.jsonl"
MB_OUTPUT = f"/notebooks/CogMem/results/memory_bank_full_{MODEL.replace(':', '_').replace('-', '_')}.json"

episodes = []
with open(CHECKPOINT, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            episodes.append(json.loads(line))

Path(MB_OUTPUT).parent.mkdir(parents=True, exist_ok=True)
with open(MB_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(episodes, f, indent=2, ensure_ascii=False)

passed = sum(1 for ep in episodes if ep["success"])
print(f"Memory bank saved: {MB_OUTPUT}")
print(f"  {len(episodes)} episodes, {passed} passed ({passed/len(episodes):.1%})")

In [ ]:
# Cell 9: Show memory bank contents
import json

with open(MB_OUTPUT, encoding="utf-8") as f:
    bank = json.load(f)

print(f"{'='*70}")
print(f"MEMORY BANK: {MB_OUTPUT}")
print(f"{'='*70}")
print(f"Total episodes: {len(bank)}")
print()

print(f"{'Task ID':<25} {'Q':>5} {'Success':>8} {'CodeLen':>9} {'Error Type':<20}")
print(f"{'-'*70}")

for ep in bank:
    task_id = ep.get("task_id", "?")
    q = ep.get("q_value", 0)
    success = "PASS" if ep.get("success") else "FAIL"
    code_len = len(ep.get("generated_code", ""))
    
    err_type = ""
    if not ep.get("success") and ep.get("error"):
        err = ep["error"][:100]
        if "Timeout" in err: err_type = "Timeout"
        elif "SyntaxError" in err: err_type = "SyntaxError"
        elif "ImportError" in err or "ModuleNotFoundError" in err: err_type = "ImportError"
        elif "NameError" in err: err_type = "NameError"
        elif "TypeError" in err: err_type = "TypeError"
        else: err_type = "RuntimeError"
    
    print(f"{task_id:<25} {q:>5.1f} {success:>8} {code_len:>8}c {err_type:<20}")

print(f"\n{'='*70}")
q_vals = [ep["q_value"] for ep in bank]
passed = sum(1 for ep in bank if ep["success"])
print(f"Summary: {passed}/{len(bank)} passed | "
      f"Q mean={sum(q_vals)/len(q_vals):.2f} | "
      f"High(>=0.7)={sum(1 for q in q_vals if q>=0.7)} | "
      f"Low(<0.3)={sum(1 for q in q_vals if q<0.3)}")